In [19]:
pip install scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.


# AGRUPAMIENTO DE DOCUMENTOS
- Clustering
- Analizar resultados (En base a a la efectividad y peculiaridad del algoritmo)

## ¿Que hay que hacer?
- Hay que aplicar el algoritmo k-means sobre las 3 representaciones

|| Semilla 1 | Semilla 2 | Semilla 3 |
|-----------|-----------|-----------|--|
| Binario   | Dato      | Dato      | |
| TF        | Dato      | Dato      | |
| TF-IDF    | Dato      | Dato      | |

- Además hay que hacer con  TF-IDF aplicar el algoritmo EM para compararlo con K-Means
- Utilizamos t-SNE para la representación de los datos

# CLASIFICACION DE DOCUMENTOS
- Clasificación
- Analisis detallados (Precisión, eficacia y limitaciones de métodos aplicados)

## ¿Que hay que hacer?
- Hay que aplicar el algoritmo k-NN sobre las 3 representaciones. Combinaciones de K, esquema de pesos y valor p de la distancia de minkowski

|| Combo 1 | Combo 2 | Combo 3 | Combo 4 | Combo 5 | 
|-----------|-----------|-----------|--|--|--|
| Binario   | Dato      | Dato      | | | |
| TF        | Dato      | Dato      | | | |
| TF-IDF    | Dato      | Dato      | | | |

- Además hay que hacer Naive Bayes con cada representación

|| Gausiana  | Multinomial 2 |
|-----------|-----------|-----------|
| Binario   | Dato      | Dato      |  
| TF        | Dato      | Dato      | 
| TF-IDF    | Dato      | Dato      | 

- Comparar los resultados obtenido por K-NN y Naive Bayes y dteminer cual es la mejor representación atendiendo a la precisión

# COMPETICIÓN DE KAGGLE

# Código
## Preprocesamiento
- Eliminar stopwords
- stemming
- Otras metodologías (Como cuales)

## Representación
- Binaria
- TF
- TF-IDF

Hay que evaluar como cada enfoque afecta a la calidad de análisis y el conocimiento extraido de los docs.

## Estudio con validación estratificada cruzada
- Divides en 5 y haces la combinatoria
- Mirar la doc de StratifiedKFold en stikit-learn

## Dataset
- Tenemos que decidir, qué categorias son relevantes y porqué.

### Enlaces
- Trabajar con datos de Texto en scikit-learn: https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html
- Visualizar clusters con t-SNE: https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html

In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold

print("Librerías importadas correctamente")


ModuleNotFoundError: No module named 'pandas'

In [6]:
dataset = 'news_reducido.csv'

# Leer los datos en formato csv
data = pd.read_csv(dataset, on_bad_lines='skip')
# Nos quedamos con el texto (puedes quedarte con más información si quieres)
X = data['text'].fillna('').astype(str).to_numpy() # Ponco con un espacio los espacio nulos

NameError: name 'pd' is not defined

In [24]:
# Ahora, procesamos las etiquetas, para cada clase, le asignamos un valor numérico entre 0 y el número de clases
semilla1=42
semilla2=640
semilla3=5300742

semilla = semilla1

enc = OrdinalEncoder()
y = enc.fit_transform(np.reshape(data['category'], (-1,1))).reshape(-1)

# Hacemos la partición train-test con Validacion cruzada estratificada
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=semilla)
skf.get_n_splits(X, y)

# Definir aquí los pipelines necesarios para cada problema (clustering, clasificación, etc.)
text_kmeans_binario = Pipeline([
    ('vect', CountVectorizer(binary=True)),
    ('clf', KMeans(n_clusters=4))
])

text_kmeans_tf = Pipeline([
    ('vect', CountVectorizer(binary=False)),
    ('clf', KMeans(n_clusters=4))
])

text_kmeans_tfidf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', KMeans(n_clusters=4))
])

# Clasificación
text_sgd = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB()),
])

text_sgd = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB()),
])

text_kmeans = text_kmeans_binario


In [ ]:
accuracies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X,y)):

    fit_clustering = True
    # Clustering
    if fit_clustering:
        # Entrenamiento
        text_kmeans.fit(X[tra])
        # Test
        labels = text_kmeans.predict(X[tst])
        # Calculo de metricas
        acc = np.mean(labels == y[tst])
        print(acc)
        accuracies[i] = acc

# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

In [25]:
# Este te saca las cosas con las fotos
accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
        
    fit_clustering = True
    # Clustering
    if fit_clustering:
        # Entrenamiento
        text_kmeans.fit(X[tra])
        # Test
        labels = text_kmeans.predict(X[tst])
        # Calculo de metricas
        acc = np.mean(labels == y[tst])
        print(acc)
        accuracies[i] = acc

        # ===== t-SNE SOLO del test de este fold =====
        X_tst_counts = text_kmeans.named_steps['vect'].transform(X[tst])
        X_tst_tfidf = text_kmeans.named_steps['tfidf'].transform(X_tst_counts)

        tsne = TSNE(
            n_components=2,
            perplexity=min(30, max(5, len(X[tst]) // 3)),
            learning_rate='auto',
            init='pca',
            random_state=42
        )

        tst_2d = tsne.fit_transform(X_tst_tfidf)

        plt.figure(figsize=(9, 6))
        plt.scatter(tst_2d[:, 0], tst_2d[:, 1], c=labels, s=12, cmap='tab10', alpha=0.8)
        plt.title(f"t-SNE fold {i+1} (color = cluster predicho en tst)")
        plt.xlabel("t-SNE 1")
        plt.ylabel("t-SNE 2")
        plt.colorbar(label="Cluster")
        plt.tight_layout()
        plt.savefig(f"./tsne_fold_{i+1}_clusters_tst.png", dpi=300, bbox_inches="tight")
        plt.close()

# Tras el K-Fold, hay que mostrar la precision media obtenida ( o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

0.159
0.1565
0.571
0.2115
0.123
Precision media = 0.24419999999999997


In [21]:
accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    fit_classification = False
    # Clasificacion
    if fit_classification:
        # Entrenamientoo
        text_sgd.fit(X[tra], y[tra])

        # Test (obtener predicciones)
        predicted = text_sgd.predict(X[tst])

        # Calculo de metricas de calidad (ahora, solo accuracy)
        acc = np.mean(predicted == y[tst])

        print(acc)
        accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida ( o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

Precision media = 0.0
